In [3]:
from faker import Faker
import random
import json
import pandas as pd

fake = Faker('en_IN')

In [5]:
eci_df = pd.read_csv("10_Voters_Information_4.csv")

karnataka = eci_df[
    (eci_df['State Name'] == 'Karnataka') & 
    (eci_df['Constituency Type'] == 'Total')
].iloc[0]

total_electors = karnataka['Electors - Male'] + karnataka['Electors - Female']
MALE_RATIO = karnataka['Electors - Male'] / total_electors
FEMALE_RATIO = karnataka['Electors - Female'] / total_electors
TURNOUT = karnataka['Voters - Poll %'] / 100

print(f"Karnataka Male Ratio: {MALE_RATIO:.4f}")
print(f"Karnataka Female Ratio: {FEMALE_RATIO:.4f}")
print(f"Karnataka Voter Turnout: {TURNOUT:.4f}")

Karnataka Male Ratio: 0.5005
Karnataka Female Ratio: 0.4995
Karnataka Voter Turnout: 0.7090


In [6]:
CONSTITUENCIES = [
    "Bangalore North", "Bangalore South",
    "Bangalore East", "Bangalore West",
    "Bangalore Central", "Yelahanka"
]

BOOTHS = {
    "Bangalore North":   ["BN-B01", "BN-B02", "BN-B03"],
    "Bangalore South":   ["BS-B01", "BS-B02", "BS-B03"],
    "Bangalore East":    ["BE-B01", "BE-B02", "BE-B03"],
    "Bangalore West":    ["BW-B01", "BW-B02", "BW-B03"],
    "Bangalore Central": ["BC-B01", "BC-B02", "BC-B03"],
    "Yelahanka":         ["YL-B01", "YL-B02", "YL-B03"],
}

PARTS = {
    "BN-B01": 1, "BN-B02": 2, "BN-B03": 3,
    "BS-B01": 4, "BS-B02": 5, "BS-B03": 6,
    "BE-B01": 7, "BE-B02": 8, "BE-B03": 9,
    "BW-B01": 10, "BW-B02": 11, "BW-B03": 12,
    "BC-B01": 13, "BC-B02": 14, "BC-B03": 15,
    "YL-B01": 16, "YL-B02": 17, "YL-B03": 18,
}

In [9]:
def generate_voter_id():
    # Real EPIC format: 3 uppercase letters + 7 digits
    # Prefix is a constituency/state code — not a readable abbreviation
    letters = fake.lexify("???").upper()   # 3 random uppercase letters
    digits = fake.numerify("#######")      # 7 random digits
    return f"{letters}{digits}"

In [10]:
def generate_phone():
    first_digit = random.choice(["6", "7", "8", "9"])
    remaining = fake.numerify("#########")
    return f"+91 {first_digit}{remaining[:4]} {remaining[4:]}"

In [11]:
def generate_voters(count=1000):
    voters = []
    for _ in range(count):
        constituency = random.choice(CONSTITUENCIES)
        booth = random.choice(BOOTHS[constituency])
        part = PARTS[booth]

        gender = random.choices(
            ["Male", "Female"],
            weights=[MALE_RATIO, FEMALE_RATIO]
        )[0]

        if gender == "Male":
            name = fake.name_male()
            relative_name = fake.name_male()
            relative_type = random.choice(["Father", "Husband"])
        else:
            name = fake.name_female()
            relative_name = fake.name_male()
            relative_type = random.choice(["Father", "Husband"])

        voter = {
            "voter_id":      generate_voter_id(),
            "name":          name,
            "relative_name": relative_name,
            "relative_type": relative_type,
            "dob":           fake.date_of_birth(
                                 minimum_age=18,
                                 maximum_age=80
                             ).strftime("%Y-%m-%d"),
            "gender":        gender,
            "phone":         generate_phone(),
            "address":       fake.address().replace("\n", ", "),
            "constituency":  constituency,
            "booth_id":      booth,
            "part_number":   part,
            "has_voted":     False,
        }
        voters.append(voter)
    return voters

voters = generate_voters(1000)
print(f"Generated {len(voters)} voters!")
print(f"Male: {sum(1 for v in voters if v['gender'] == 'Male')}")
print(f"Female: {sum(1 for v in voters if v['gender'] == 'Female')}")

Generated 1000 voters!
Male: 502
Female: 498


In [ ]:
df = pd.DataFrame(voters)
df.head(5)

,voter_id,name,relative_name,relative_type,dob,gender,phone,address,constituency,booth_id,part_number,has_voted
995,CFG1260921,Yutika Narula,Logan Mammen,Husband,1959-06-10,Female,+91 73891 74673,"82, Keer Circle, Kishanganj 974956",Bangalore South,BS-B01,4,False
996,KQA2073162,Charvi Mody,Gagan Gaba,Husband,1957-05-27,Female,+91 97435 78045,"35/61, Murty Chowk, Ballia-353378",Bangalore North,BN-B03,3,False
997,UBI6527894,Bishakha Chandra,Gabriel Reddy,Husband,1966-11-30,Female,+91 86812 95817,"H.No. 42, Mandal Ganj, Kanpur 798074",Bangalore West,BW-B03,12,False
998,DZB9575961,Mohini Bobal,Neel Mohanty,Father,1976-12-22,Female,+91 63675 67133,"18, Deo Zila, Bahraich-835133",Bangalore East,BE-B02,8,False
999,HBO6876636,Ekantika Vig,Gabriel Kapur,Husband,2002-08-03,Female,+91 73396 88484,"H.No. 989, Gulati Street, Jamalpur 351957",Bangalore East,BE-B01,7,False


In [ ]:
my_record = {
    "voter_id":      "XKP1234567",
    "name":          "Monika",
    "relative_name": "Ranga Raju",
    "relative_type": "Father",
    "dob":           "2000-01-01",
    "gender":        "Female",
    "phone":         "+91 98765 43210",
    "address":       "Hanumanth Nagar, Bangalore, Karnataka",
    "constituency":  "Bangalore South",
    "booth_id":      "BS-B02",
    "part_number":   13,
    "has_voted":     False,
}

voters.append(my_record)
print(f"Total voters including you: {len(voters)}")

Total voters including you: 1001


In [ ]:
with open("voters.json", "w") as f:
    json.dump(voters, f, indent=2)

print("Saved to voters.json!")
print(f"\nYour record:")
print(json.dumps(voters[-1], indent=2))

Saved to voters.json!

Your record:
{
  "voter_id": "XKP1234567",
  "name": "Monika",
  "relative_name": "Ranga Raju",
  "relative_type": "Father",
  "dob": "2000-01-01",
  "gender": "Female",
  "phone": "+91 98765 43210",
  "address": "Hanumanth Nagar, Bangalore, Karnataka",
  "constituency": "Bangalore South",
  "booth_id": "BS-B02",
  "part_number": 13,
  "has_voted": false
}
